# NeuroGraph · Phase 5 · Study graph (Neo4j AuraDB)
Settings: Accelerator **None**, Internet **On**.
Inputs (**Add Input → Your Work**): **neurograph-20-preprocess (Version 2)**, **neurograph-30-features (Version 3)**, **neurograph-40-models (Version 4)**, **neurograph-42-channels (latest successful version)**.
Secrets (**Add-ons → Secrets**, attached to this notebook): `NEO4J_URI`, `NEO4J_USERNAME`, `NEO4J_PASSWORD`.

## 1. Protect preinstalled packages

In [ ]:
from importlib.metadata import version

def write_constraints(pkgs: list[str], path: str) -> None:
    with open(path, "w") as f:
        f.write("\n".join(f"{p}=={version(p)}" for p in pkgs))

write_constraints(["torch", "numpy", "scipy", "scikit-learn", "mne"], "/tmp/keep.txt")
print(open("/tmp/keep.txt").read())

## 2. Get the latest code + install

In [ ]:
import os
R = "/kaggle/working/neurograph"
if not os.path.isdir(R):
    !git clone -q https://github.com/stuBBornButterfly/neurograph.git {R}
!git -C {R} pull -q
!git -C {R} log --oneline -1
%cd {R}
!pip install -q -r requirements.txt -c /tmp/keep.txt
!pip install -q -e . -c /tmp/keep.txt

## 3. Check the inputs

In [ ]:
!find /kaggle/input -name qc_table.csv
!find /kaggle/input -name features.csv
!find /kaggle/input -name risk_tiers.csv
!find /kaggle/input -name all_results.csv

## 4. Load the AuraDB credentials from Kaggle Secrets (values are never printed)

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
for key in ["NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD"]:
    os.environ[key] = secrets.get_secret(key)
print("loaded:", [k for k in ["NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD"] if os.environ.get(k)])

## 5. Step 5.1: load the study graph

In [ ]:
!cd /kaggle/working/neurograph && python -m scripts.build_study_graph

## 6. Step 5.2: ten Cypher queries

In [ ]:
!cd /kaggle/working/neurograph && python -m scripts.run_graph_queries